In [1]:
from googleapiclient.discovery import build
import re
import pandas as pd
import os
from dotenv import load_dotenv

In [ ]:
# Load API key from file .env
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")

youtube = build('youtube', 'v3', developerKey=API_KEY)

In [4]:
def clean_newlines(text):
    return " ".join(text.split()).strip()

def remove_emojis(text):
    emoji_pattern = re.compile(
        "["  
        "\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F300-\U0001F5FF"  # Symbols & pictographs
        "\U0001F680-\U0001F6FF"  # Transport & map
        "\U0001F1E0-\U0001F1FF"  # Flags
        "\U00002700-\U000027BF"  # Dingbats
        "\U000024C2-\U0001F251"  # Enclosed
        "]+", flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', text)

def preprocess_comment(text):
    text = clean_newlines(text)
    text = remove_emojis(text)
    return text.strip()

def get_long_comments(video_id, min_words=15, max_results=100):
    next_page_token = None
    comments = []

    while True:
        response = youtube.commentThreads().list(
            part='snippet',
            videoId=video_id,
            maxResults=max_results,
            pageToken=next_page_token,
            textFormat='plainText'
        ).execute()

        for item in response['items']:
            comment_text = item['snippet']['topLevelComment']['snippet']['textDisplay']
            comment_text = preprocess_comment(comment_text)
            if len(comment_text.split()) >= min_words:
                comments.append(comment_text)

        if 'nextPageToken' in response and len(comments) < max_results:
            next_page_token = response['nextPageToken']
        else:
            break

    return comments

In [5]:
video_ids = ['eHueOuMZ69c']
long_comments = []

for vid in video_ids:
    print(f"Đang lấy comment dài cho video: {vid}")
    new_comments = get_long_comments(vid)
    long_comments.extend(new_comments)
    print(f"Đã lấy {len(new_comments)} comment dài cho video {vid}")

# Xuất ra CSV
df = pd.DataFrame(long_comments, columns=["comment"])
df.to_csv("youtube_comments.csv", index=False, encoding="utf-8-sig")

Đang lấy comment dài cho video: eHueOuMZ69c
Đã lấy 95 comment dài cho video eHueOuMZ69c
